# Zor test ölçümü: Model A,Model B,Model C ve Model D 

Gerçek kurumsal SMS'lerle iki eğitim senaryosunu karşılaştırır (bkz. `docs/zor-test.md`).
Veri `data/external/` altındadır ve git'e girmez. 

In [1]:
import sys
sys.path.append("..")
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from src.temizle import Temizleyici

svm = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2,
                            sublinear_tf=True, lowercase=False)),
    ("model", LinearSVC(class_weight="balanced", random_state=55)),
])

In [2]:
# Eğitim verisi
train = pd.read_csv("../data/processed/tr_train.csv")
train = train[train["belirsiz"] == "h"].reset_index(drop=True)
X = train["mesaj"]
y = (train["ikili"] == "spam").astype(int)

# Zor test verisi
zor = pd.read_csv("../data/external/zor_gercek_hepsi.csv", keep_default_na=False)
zor = zor[zor["etiket"] != ""].reset_index(drop=True)      # belirsizleri at
y_zor = (zor["etiket"] != "meşru").astype(int)             # spam = reklam + dolandırıcılık

print(len(X), len(zor))
print(zor["etiket"].value_counts().to_dict())

651 131
{'meşru': 67, 'reklam': 54, 'dolandırıcılık': 10}


In [3]:
kisim = pd.Series(["1. kısım"] * 84 + ["2. kısım"] * (len(zor) - 84), index=zor.index)
print(zor.loc[kisim == "1. kısım", "etiket"].value_counts().to_dict())   # meşru 49, reklam 24, dolandırıcılık 11

{'meşru': 48, 'reklam': 26, 'dolandırıcılık': 10}


Model A’da zor mesajlar eğitime hiç girmiyor, sadece sınav

In [8]:
kisim = pd.Series(["1. kısım"] * 82 + ["2. kısım"] * (len(zor) - 82), index=zor.index)
print(zor.loc[kisim == "1. kısım", "etiket"].value_counts().to_dict())   # meşru 48, reklam 24, dolandırıcılık 10

def rapor(tahmin, ad):
    tahmin = pd.Series(tahmin, index=zor.index)
    for kisim_adi, maske in [("1. kısım", kisim == "1. kısım"),
                             ("2. kısım", kisim == "2. kısım"),
                             ("birleşik", kisim.notna())]:
        gercek = y_zor[maske]
        t = tahmin[maske]
        yanlis = ((t == 1) & (gercek == 0)).sum()
        yakalanan = ((t == 1) & (gercek == 1)).sum()
        print(f"{ad} | {kisim_adi:9s} | spam F1: {f1_score(gercek, t):.2f} | "
              f"yanlış alarm: {yanlis}/{(gercek == 0).sum()} | "
              f"yakalanan spam: {yakalanan}/{(gercek == 1).sum()}")

# Model A: yalnızca tr_train ile eğitilmiş
tahmin_A = clone(svm).fit(X, y).predict(zor["mesaj"])
rapor(tahmin_A, "Model A")

{'meşru': 48, 'reklam': 24, 'dolandırıcılık': 10}
Model A | 1. kısım  | spam F1: 0.66 | yanlış alarm: 27/48 | yakalanan spam: 30/34
Model A | 2. kısım  | spam F1: 0.90 | yanlış alarm: 7/19 | yakalanan spam: 30/30
Model A | birleşik  | spam F1: 0.76 | yanlış alarm: 34/67 | yakalanan spam: 60/64


Model A’de gonderen sütununa göre dönüyoruz, her turda bir göndericiyi sınav yapıp kalanları eğitime ekliyoruz:

In [10]:
grup = zor["gonderen"].str.upper()
tahmin_B = pd.Series(0, index=zor.index)

for g in grup.unique():
    sinav = grup == g
    egit_X = pd.concat([X, zor.loc[~sinav, "mesaj"]], ignore_index=True)
    egit_y = pd.concat([y, y_zor[~sinav]], ignore_index=True)
    model = clone(svm).fit(egit_X, egit_y)
    tahmin_B[sinav] = model.predict(zor.loc[sinav, "mesaj"])

rapor(tahmin_B, "Model B")

Model B | 1. kısım  | spam F1: 0.83 | yanlış alarm: 10/48 | yakalanan spam: 31/34
Model B | 2. kısım  | spam F1: 0.92 | yanlış alarm: 3/19 | yakalanan spam: 28/30
Model B | birleşik  | spam F1: 0.87 | yanlış alarm: 13/67 | yakalanan spam: 59/64


In [15]:
import os
print(sorted(os.listdir("../data/sentetik")))

['ceviri_v1.csv', 'dolandiricilik_v1.csv', 'sentetik_dolandiricilik_v1.csv', 'sentetik_kurumsal_mesru_v1.csv', 'sentetik_kurumsal_mesru_v2.csv']


In [16]:
def dolandiricilik(tahmin, ad):
    d = (zor["etiket"] == "dolandırıcılık")
    t = pd.Series(tahmin, index=zor.index)
    print(ad, "| dolandırıcılık yakalama:", (t[d] == 1).sum(), "/", d.sum())

In [ ]:
import glob

sent = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("../data/sentetik/sentetik_*.csv"))],
                 ignore_index=True)
y_sent = (sent["etiket"] != "meşru").astype(int)
print(len(sent), sent["etiket"].value_counts().to_dict())

88 {'meşru': 48, 'dolandırıcılık': 40}


In [18]:
# Model C: tr_train + sentetik
X_c = pd.concat([X, sent["mesaj"]], ignore_index=True)
y_c = pd.concat([y, y_sent], ignore_index=True)
tahmin_C = clone(svm).fit(X_c, y_c).predict(zor["mesaj"])
rapor(tahmin_C, "Model C")

Model C | 1. kısım  | spam F1: 0.75 | yanlış alarm: 21/48 | yakalanan spam: 33/34
Model C | 2. kısım  | spam F1: 0.89 | yanlış alarm: 5/19 | yakalanan spam: 28/30
Model C | birleşik  | spam F1: 0.81 | yanlış alarm: 26/67 | yakalanan spam: 61/64


In [19]:
# Model D: tr_train + sentetik + diğer göndericilerin gerçek mesajları
grup = zor["gonderen"].str.upper()
tahmin_D = pd.Series(0, index=zor.index)
for g in grup.unique():
    sinav = grup == g
    egit_X = pd.concat([X, sent["mesaj"], zor.loc[~sinav, "mesaj"]], ignore_index=True)
    egit_y = pd.concat([y, y_sent, y_zor[~sinav]], ignore_index=True)
    tahmin_D[sinav] = clone(svm).fit(egit_X, egit_y).predict(zor.loc[sinav, "mesaj"])
rapor(tahmin_D, "Model D")

Model D | 1. kısım  | spam F1: 0.85 | yanlış alarm: 9/48 | yakalanan spam: 32/34
Model D | 2. kısım  | spam F1: 0.93 | yanlış alarm: 2/19 | yakalanan spam: 28/30
Model D | birleşik  | spam F1: 0.89 | yanlış alarm: 11/67 | yakalanan spam: 60/64


In [20]:
dolandiricilik(tahmin_A, "Model A")
dolandiricilik(tahmin_B, "Model B")
dolandiricilik(tahmin_C, "Model C")
dolandiricilik(tahmin_D, "Model D")

Model A | dolandırıcılık yakalama: 8 / 10
Model B | dolandırıcılık yakalama: 9 / 10
Model C | dolandırıcılık yakalama: 9 / 10
Model D | dolandırıcılık yakalama: 10 / 10


## Sonuçlar (çıktılar temizlendiği için buraya yazıldı)

Etiketli mesaj: 131 (67 meşru, 54 reklam, 10 dolandırıcılık). 1. kısım 82, 2. kısım 49.
Sentetik: 88 mesaj (48 kurumsal meşru, 40 kurum taklidi dolandırıcılık), yalnızca eğitimde.

| Model | Eğitimde | Birleşik F1 | Yanlış alarm | Yakalanan spam | Yakalanan dolandırıcılık |
|---|---|---|---|---|---|
| A | `tr_train` | 0.76 | 34/67 | 60/64 | 8/10 |
| C | `tr_train` + sentetik | 0.81 | 26/67 | 61/64 | 9/10 |
| B | `tr_train` + diğer göndericiler (gerçek) | 0.87 | 13/67 | 59/64 | 9/10 |
| D | `tr_train` + sentetik + diğer göndericiler | 0.89 | 11/67 | 60/64 | 10/10 |

B ve D farkı (iki mesaj) gürültü düzeyindedir (±0.02). Gönderici adları büyük harfe
normalize edilerek gruplandı.